# Project Style LoRA Colab\n\n這份 notebook 採用 Colab-first 工作流：\n- dataset 先在 `/content` 本地準備\n- 訓練也在 `/content` 執行\n- 最後再同步回 Google Drive\n\n這樣會比直接在 Google Drive 上做大量小檔複製穩得多。

## 1. 安裝

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive', force_remount=False)\n\n!pip install --upgrade pip\n!pip install "accelerate>=0.34.0" "diffusers>=0.35.0" "transformers>=4.44.0" "huggingface-hub>=0.34.0" "safetensors>=0.4.4" "Pillow>=10.4.0" "PyYAML>=6.0.2"\n\nimport os\nrepo_dir = '/content/AI_Card_Project/workspace/AI_card_excercise'\nif not os.path.exists(repo_dir):\n    !git clone https://github.com/shu2891/AI_card_excercise.git {repo_dir}\nelse:\n    print('repo already exists:', repo_dir)

## 2. 設定專題

In [ ]:
import sys\nfrom pathlib import Path\n\nREPO_DIR = Path('/content/AI_Card_Project/workspace/AI_card_excercise')\nif str(REPO_DIR) not in sys.path:\n    sys.path.insert(0, str(REPO_DIR))\nif str(REPO_DIR / 'src') not in sys.path:\n    sys.path.insert(0, str(REPO_DIR / 'src'))\n\nfrom colab_style_lora_workflow import resolve_project_paths, ensure_project_dirs, reset_runtime_dirs\n\nPROJECT_NAME = '翻身'\nSTYLE_TOKEN = 'fishseriesstyle'\nTRIGGER_PHRASE = 'fish series style'\nBASE_MODEL = 'stabilityai/stable-diffusion-xl-base-1.0'\nMODEL_NAME = 'Quality (SDXL Base)'\nPROMPT = 'fishseriesstyle, same series, pastel chalk fish, single fish hero, mythic regal power'\nSEED = 42\nNUM_IMAGES = 4\nWIDTH = 832\nHEIGHT = 1216\nLORA_SCALE = 0.95\n\npaths = resolve_project_paths(project_name=PROJECT_NAME)\nensure_project_dirs(paths)\nreset_runtime_dirs(paths)\nprint(paths)

## 3. 準備資料

In [ ]:
from colab_style_lora_workflow import prepare_style_dataset, summarize_style_dataset\n\nrecords = prepare_style_dataset(\n    source_dir=paths.source_dir,\n    dataset_dir=paths.dataset_dir,\n    style_token=STYLE_TOKEN,\n    trigger_phrase=TRIGGER_PHRASE,\n    repeats=4,\n    runtime_dataset_dir=paths.runtime_dataset_dir,\n    sync_to_drive=True,\n)\n\nsummary = summarize_style_dataset(records)\nsummary

In [ ]:
from pathlib import Path\nimport json\n\nmetadata_path = Path(paths.runtime_dataset_dir) / 'metadata.jsonl'\nprint('runtime metadata exists =', metadata_path.exists())\nprint('runtime image count =', len(list((Path(paths.runtime_dataset_dir) / 'images').glob('*'))))\n\nwith metadata_path.open('r', encoding='utf-8') as f:\n    for i, line in enumerate(f):\n        print(json.loads(line))\n        if i >= 2:\n            break

## 4. 訓練

In [ ]:
from colab_style_lora_workflow import train_style_lora\n\ntrain_style_lora(\n    paths=paths,\n    base_model=BASE_MODEL,\n    resolution=512,\n    train_batch_size=1,\n    gradient_accumulation_steps=2,\n    learning_rate=5e-5,\n    max_train_steps=300,\n    rank=8,\n    checkpointing_steps=100,\n    mixed_precision='fp16',\n    sync_to_drive=True,\n)

## 5. 生成

In [ ]:
from colab_style_lora_workflow import latest_lora_path, generate_images\n\nactive_lora = latest_lora_path(paths.runtime_lora_dir if paths.runtime_lora_dir.exists() else paths.lora_dir)\nsaved_images = generate_images(\n    prompt=PROMPT,\n    seed=SEED,\n    num_images=NUM_IMAGES,\n    lora_path=active_lora,\n    output_dir=paths.output_dir,\n    model_name=MODEL_NAME,\n    width=WIDTH,\n    height=HEIGHT,\n    lora_scale=LORA_SCALE,\n    runtime_output_dir=paths.runtime_output_dir,\n    sync_to_drive=True,\n)\nsaved_images